PART 02：注册表 + 触发器——挂钩系统本体不到二十行
先看全景。挂钩系统就三样东西：一张注册表、一个注册函数、一个触发函数。

In [ ]:
HOOKS = {
    "UserPromptSubmit": [],   # 用户输入提交后、进 LLM 之前
    "PreToolUse":      [],    # 工具执行之前
    "PostToolUse":     [],    # 工具执行之后
    "Stop":            [],    # 循环即将退出时
}

def register_hook(event: str, callback):
    HOOKS[event].append(callback)

def trigger_hooks(event: str, *args):
    for callback in HOOKS[event]:
        result = callback(*args)
        if result is not None:      # 返回非 None → 有话说，立即中断
            return result
    return None

没了。一张字典，事件名映射到回调列表；注册就是往列表里 append；触发就是遍历着挨个执行。

但有一行值得你停下来盯十秒，它藏着整个系统的灵魂：

In [ ]:
if result is not None:
    return result

这个约定读出来是一句话：回调返回 None，表示"我没意见"，继续走；返回任何非 None 的东西，表示"我有话说"，立刻拦下，把这句话带出去。

现在它只是个奇怪的默认值。到这一篇的高潮你会看到，这行代码能把一个"说做完了"的 Agent 拽回来继续干活。先记着。

循环的改动：拆一行，装四行
挂钩系统怎么接进循环？看改造后的 agent_loop，我把新增的行标了出来：

In [ ]:
def agent_loop(messages: list):
    while True:
        response = client.messages.create(
            model=MODEL, system=SYSTEM, messages=messages,
            tools=TOOLS, max_tokens=8000,
        )
        messages.append({"role": "assistant", "content": response.content})

        tool_calls = [b for b in response.content if b.type == "tool_use"]

        if not tool_calls:                          # 模型不再调工具，想收工
            force = trigger_hooks("Stop", messages) # ← 新增：下班前，挂钩过一遍
            if force:
                messages.append({"role": "user", "content": force})
                continue                             #   挂钩有话说 → 注入，继续转
            return                                  #   挂钩没意见 → 真下班

        results = []
        for block in tool_calls:
            blocked = trigger_hooks("PreToolUse", block)   # ← 改动：替代 check_permission
            if blocked:
                results.append({"type": "tool_result",
                                "tool_use_id": block.id,
                                "content": str(blocked)})
                continue

            handler = TOOL_HANDLERS.get(block.name)
            output = handler(**block.input) if handler else f"Unknown: {block.name}"

            trigger_hooks("PostToolUse", block, output)    # ← 新增：执行完，挂钩过一遍

            results.append({"type": "tool_result",
                            "tool_use_id": block.id, "content": output})

        messages.append({"role": "user", "content": results})

主入口那边还有一处，用户输入刚提交、还没进模型：

In [ ]:
query = input(">> ")
trigger_hooks("UserPromptSubmit", query)      # ← 新增：进 LLM 之前
history.append({"role": "user", "content": query})

数一下循环本体的净变化：删掉一行 if not check_permission(block)，换成一行 trigger_hooks("PreToolUse", block)；另外加了两行触发。心脏没有长任何功能，只是多了几个"挂钩位"。

权限去哪了？搬家，不是装修
有件事必须说清楚，不然你以为我把上一篇的门禁拆了。

三道闸门一行没删。check_deny_list、check_rules、ask_user 原封不动，只是原来在循环里直接调用，现在包成一个 hook：

In [ ]:
def permission_hook(block):
    if not check_permission(block):        # 上一篇的三道闸门，原样在里面
        return "Permission denied."        # 非 None → 拦下，拒绝回流给模型
    return None                            # None → 放行

register_hook("PreToolUse", permission_hook)

搬家，不是装修。逻辑一行没改，改的只有位置：从"心脏里面"搬到"挂钩上面"。

搬完之后发生了一个微妙但关键的变化：循环不再认识"权限"这个概念了。它不知道挂在 PreToolUse 上的是权限检查还是日志，不知道拦下模型的字符串是 deny list 发的火还是用户按的 N——它只认识"挂钩"这个抽象。谁挂上来、干什么，与循环无关。

这就是解耦的全部含义：核心只知道插口在哪，不知道插件是什么。

